# Восстановление параметров распределения для региональных данных

In [109]:
import os
import pandas as pd
from tqdm import tqdm

## 1. Подготовка таблицы с данными

In [6]:
POVERTY_FILE = './data/poverty.xlsx'
LINE_FILE = './data/poverty_line.xlsx'
INCOME_FILE = './data/average_income.xlsx'

In [92]:
poverty = pd.read_excel(POVERTY_FILE, sheet_name='по РФ и субъектам РФ 1995-2024', skiprows=2, dtype='string')
poverty_line = pd.read_excel(LINE_FILE, sheet_name='по субъектам РФ 2024', skiprows=3, dtype='string')
income = pd.read_excel(INCOME_FILE, sheet_name='Лист1', skiprows=2, dtype='string')

In [93]:
poverty = poverty[['Unnamed: 0', 2024]][poverty[2024].notna()]
poverty =poverty.rename(columns={'Unnamed: 0': 'region', 2024: 'poverty'})
poverty['region'] = poverty['region'].str.replace('в том числе:', '', regex=False).str.replace('\n', '', regex=False).str.strip()

In [94]:
poverty_line = poverty_line[['Unnamed: 0', 'Unnamed: 1']][poverty_line['Unnamed: 1'].notna()]
poverty_line = poverty_line.rename(columns={'Unnamed: 0': 'region', 'Unnamed: 1': 'poverty_line'})
poverty_line['region'] = poverty_line['region'].str.replace('в том числе:', '', regex=False).str.replace('\n', '', regex=False).str.strip()

poverty_line['region'] = poverty_line['region'].replace({
    'Ханты-Мансийский авт. округ - Югра': 'Ханты-Мансийский автономный округ - Югра',
    'Тюменская область без авт. округов': 'Тюменская область (без авт. округов)',
    'Архангельская область без авт. округа': 'Архангельская область без  авт. округа'
})

In [95]:
income = income[['Unnamed: 0', 'Unnamed: 66']][(income['Unnamed: 66'].notna()) & (income['Unnamed: 66'] != 'год')]
income = income.rename(columns={'Unnamed: 0': 'region', 'Unnamed: 66': 'income'})
income['region'] = income['region'].str.replace('в том числе:', '', regex=False).str.replace('\n', '', regex=False).str.strip()

income['region'] = income['region'].str.replace('г. ', 'Город ', regex=False)
income['region'] = income['region'].str.replace('  ', ' ', regex=False)

income['region'] = income['region'].replace({
    'Архангельская область без авт.округа': 'Архангельская область без  авт. округа',
    'Тюменская область без авт.округов': 'Тюменская область (без авт. округов)',
    'Ненецкий авт.округ': 'Ненецкий автономный округ',
    'Республика Северная Осетия - Алания': 'Республика Северная Осетия-Алания',
    'Ханты-Мансийский авт.округ': 'Ханты-Мансийский автономный округ - Югра',
    'Еврейская авт.область': 'Еврейская автономная область',
    'Ямало-Ненецкий авт.округ': 'Ямало-Ненецкий автономный округ',
    'Чукотский авт.округ': 'Чукотский автономный округ'
})


In [97]:
merged = poverty.merge(poverty_line, on='region', how='left').merge(income, on='region', how='outer')

In [103]:
merged = merged[~merged['region'].str.contains('федеральный округ', na=False)]
merged = merged[~merged['region'].isin(['Архангельская область', 'Тюменская область'])]

In [104]:
merged.to_excel('./data/merged.xlsx', index=False)

## 2. Расчет распределения доходов

In [105]:
from hack_params import HackParams

In [106]:
merged['income'] = merged['income'].astype(float)
merged['poverty'] = merged['poverty'].astype(float)
merged['poverty_line'] = merged['poverty_line'].astype(float)

In [115]:
poverty_steps_all = []

for idx, row in tqdm(merged.iterrows(), total=len(merged)):
    new_poverty = HackParams(average_income=row['income'], 
                             poverty_line=row['poverty_line'], 
                             poverty=row['poverty'])

    new_poverty.find_params()
    poverty_steps = pd.DataFrame({'sum': list(range(1000, 200001, 50))})
    poverty_steps['share'] = poverty_steps['sum'].apply(new_poverty.count_poverty)
    poverty_steps['region'] = row['region']
    poverty_steps['poverty'] = row['poverty']
    poverty_steps['poverty_line'] = row['poverty_line']
    poverty_steps['income'] = row['income']
    poverty_steps_all.append(poverty_steps)

poverty_steps_df = pd.concat(poverty_steps_all, ignore_index=True)

100%|██████████| 86/86 [01:33<00:00,  1.09s/it]


In [117]:
poverty_steps

,sum,share,region,poverty,poverty_line,income
0,1000,0.0,Ярославская область,6.7,14633.0,52198.0
1,1050,0.0,Ярославская область,6.7,14633.0,52198.0
2,1100,0.0,Ярославская область,6.7,14633.0,52198.0
3,1150,0.0,Ярославская область,6.7,14633.0,52198.0
4,1200,0.0,Ярославская область,6.7,14633.0,52198.0
...,...,...,...,...,...,...
3976,199800,98.9,Ярославская область,6.7,14633.0,52198.0
3977,199850,98.9,Ярославская область,6.7,14633.0,52198.0
3978,199900,98.9,Ярославская область,6.7,14633.0,52198.0
3979,199950,98.9,Ярославская область,6.7,14633.0,52198.0


In [116]:
poverty_steps_df.to_excel('./data/poverty_steps_df.xlsx', index=False)